# 01 — Diagnóstico del PDF
Antes de escribir un parser: ¿el PDF tiene texto o hace falta OCR? ¿qué páginas están vacías?
¿cómo se rotulan los cuadros? No escribe nada en `data/`.

In [ ]:
%run -i modulos/comun.ipynb
import re
from collections import Counter
import matplotlib.pyplot as plt

UMBRAL_VACIA = 60   # caracteres; por debajo, la página es un mapa o una imagen


def ejecutar(cmd):
    res = subprocess.run(cmd, capture_output=True, text=True, errors="replace")
    if res.returncode != 0:
        raise RuntimeError("Falló " + " ".join(cmd) + ":\n" + res.stderr)
    return res.stdout


def texto_pagina(n):
    return ejecutar(["pdftotext", "-layout", "-f", str(n), "-l", str(n), str(PDF), "-"])

## Metadatos

In [ ]:
meta = ejecutar(["pdfinfo", str(PDF)])
print(meta.strip())
n_paginas = int(re.search(r"^Pages:\s+(\d+)", meta, re.M).group(1))

## Capa de texto

In [ ]:
completo = ejecutar(["pdftotext", "-layout", str(PDF), "-"])
pags = completo.split("\f")
if len(pags) > 0 and pags[-1] == "":
    pags.pop()

caracteres = []
for p in pags:
    caracteres.append(len(p.strip()))
total_chars = sum(caracteres)

print("Páginas según pdfinfo  :", n_paginas)
print("Páginas según pdftotext:", len(pags))
print("Caracteres extraídos   :", total_chars)
print("Promedio por página    :", total_chars // len(pags))
if total_chars > 100000:
    print("Veredicto: hay capa de texto, NO se necesita OCR.")
else:
    print("Veredicto: capa de texto insuficiente, revisar.")

In [ ]:
colores = []
for c in caracteres:
    if c < UMBRAL_VACIA:
        colores.append("tab:red")
    else:
        colores.append("tab:blue")
plt.figure(figsize=(12, 3.5))
plt.bar(range(1, len(pags) + 1), caracteres, color=colores, width=1.0)
plt.xlabel("página del PDF")
plt.ylabel("caracteres")
plt.title("Texto extraído por página (en rojo, páginas casi vacías)")
plt.tight_layout()
plt.show()

## Páginas casi vacías

In [ ]:
vacias = []
for i in range(len(pags)):
    if caracteres[i] < UMBRAL_VACIA:
        vacias.append(i + 1)
print("Total:", len(vacias))
for pag in vacias:
    muestra = " ".join(pags[pag - 1].split())[:50]
    print("  pág.", pag, " ", caracteres[pag - 1], "chars ", repr(muestra))

## Literales de navegación: "Cuadro Nro." y "Gráfica Nro."

In [ ]:
pat_cuadro = re.compile(r"Cuadro\s+Nro\.", re.I)
pat_grafica = re.compile(r"Gr[áa]fica\s+Nro\.", re.I)
pags_cuadro = []
pags_grafica = []
for i in range(len(pags)):
    if pat_cuadro.search(pags[i]):
        pags_cuadro.append(i + 1)
    if pat_grafica.search(pags[i]):
        pags_grafica.append(i + 1)
print("Páginas con 'Cuadro Nro.' :", len(pags_cuadro))
print("Páginas con 'Gráfica Nro.':", len(pags_grafica))
solapan = sorted(set(pags_cuadro) & set(pags_grafica))
print("Páginas con ambos literales:", len(solapan), solapan[:20])

ids = re.findall(r"Cuadro\s+Nro\.\s*([0-9]+(?:\.[0-9]+)*)", completo, re.I)
print("IDs de cuadro capturados:", len(ids), "(", len(set(ids)), "únicos)")
repetidos = []
for k, v in Counter(ids).items():
    if v > 1:
        repetidos.append(k)
print("IDs repetidos (mismo cuadro en más de una página):", len(repetidos), sorted(repetidos)[:15])

## Muestras crudas: página 673 (cuadro 9.1.1) y 737 (cuadro 13.1.1)

In [ ]:
print(texto_pagina(673))

In [ ]:
print(texto_pagina(737)[:2500])